In [1]:
import ast
import hashlib
import json
import math
import os
import platform
import random
import sqlite3
import sys
import tempfile
import time
import types
from collections import Counter
from datetime import datetime, timezone
from io import StringIO
from numbers import Real
from pathlib import Path
from unittest.mock import patch

start_folder = Path.cwd().resolve()

project_folder = None

for folder in (start_folder, *start_folder.parents):
    for path in (folder / "app" / "flatfairproj", folder / "flatfairproj", folder):
        if (path / "manage.py").is_file():
            project_folder = path

            break

    if project_folder is not None:
        break

if project_folder is None:
    raise Exception("Run this notebook from inside the FlatFair project folder.")

root_folder = project_folder.parents[1]

training_folder = project_folder / "assets" / "training"

In [2]:
check_choice = "flats"

short_run = True

if short_run:
    report_folder = training_folder / "check_runs" / "checks"

    flat_trials = 20
else:
    report_folder = training_folder

    flat_trials = 1000

flat_database = project_folder / "db.sqlite3"

flat_seed = 2108

photo_database = project_folder / "db.sqlite3"

label_folder = training_folder / "human_labels"

label_report_file = report_folder / "human_label_eval.json"

app_label = ""

app_report_file = report_folder / "integration_run.json"

if "django_state" not in globals():
    django_state = dict(check=None)

print("Short checks" if short_run else "Full checks")
print({"check": check_choice, "random_trials": flat_trials, "output": str(report_folder)})

Short checks
{'check': 'flats', 'random_trials': 20, 'output': '/home/coco/Desktop/c_windows/Users/Koko/Desktop/HELP/25_10_2026/IS2108/flatfair/.flatfair-local/less-code-20261114T182358Z/notebook-after/app/training/check_runs/checks'}


In [3]:
def use_django(check, rerun=True):
    from django.conf import settings

    used = django_state["check"]

    if used == check and rerun:
        return False

    if used or settings.configured:
        raise Exception(
            f"Django is already set up in this kernel ({used or 'other settings'}). "
            f"Restart the kernel, then run {check}."
        )

    return True

In [4]:
def flat_setup(database, media):
    sys.path.insert(0, str(project_folder))
    settings_file = project_folder / "flatfair" / "settings.py"

    source = settings_file.read_text()

    start = source.index("DOTENV = {}")

    end = source.index("\ndef env(", start)

    source = source[:start] + "DOTENV = {}\n\n" + source[end:]

    module = types.ModuleType("flat_recs_eval_settings")

    module.__file__ = str(settings_file)

    exec(compile(source, str(settings_file), "exec"), module.__dict__)

    module.DATABASES["default"]["NAME"] = str(database)

    module.MEDIA_ROOT = Path(media)

    sys.modules[module.__name__] = module

    os.environ["DJANGO_SETTINGS_MODULE"] = module.__name__


    import django

    django.setup()

    django_state["check"] = "flats"


def flat_marks(ranks):
    found = [rank for rank in ranks if rank is not None]

    top_six = [rank for rank in found if rank <= 6]

    return {
        "hitRateAt6": round(len(top_six) / len(ranks), 6),
        "mrr": round(sum(1 / rank for rank in found) / len(ranks), 6),
        "mrrAt6": round(sum(1 / rank for rank in top_six) / len(ranks), 6),
    }


def find_rank(ids, held_out):
    if held_out in ids:
        return ids.index(held_out) + 1

    return None


def rate_flats():
    from django.contrib.auth.models import User
    from django.db.models import Prefetch
    from django.utils import timezone
    from sellerpanel.photocheck import MODEL_VERSION
    from storefront.models import hasRole
    from storefront.models import FlatView, Listing, ListingPhoto, Offer, SavedListing, ViewingBooking
    from storefront.recommend_flats import goodFlats, rankFlats, tasteProfile

    now = timezone.now()

    rng = random.Random(flat_seed)

    rankings = {"contentBased": [], "popularity": [], "random": []}

    folds = []

    buy_ids = set()

    photos = ListingPhoto.objects.filter(modelVersion=MODEL_VERSION, embedding__isnull=False)

    buy_names = ["buyer@flatfair.sg"]

    for number in range(1, 7):
        buy_names.append(f"buyer{number}@demo.flatfair.sg")

    buyers = User.objects.filter(username__in=buy_names).order_by("pk")

    for buyer in buyers:
        if not hasRole(buyer, "buyer"):
            continue

        engaged = set(FlatView.objects.filter(buyer=buyer, count__gt=0).values_list("listing_id", flat=True))

        engaged.update(SavedListing.objects.filter(collection__user=buyer).values_list("listing_id", flat=True))

        engaged.update(Offer.objects.filter(buyer=buyer).values_list("listing_id", flat=True))

        booked = ViewingBooking.objects.filter(buyer=buyer).exclude(status="cancelled")

        engaged.update(booked.values_list("slot__listing_id", flat=True))

        engaged = set(Listing.objects.filter(pk__in=engaged).exclude(seller=buyer).values_list("pk", flat=True))

        if len(engaged) < 3:
            continue

        spares = Listing.objects.public().filter(status="live", pk__in=engaged).exclude(seller=buyer).order_by("pk")

        for spare in spares:
            profile = tasteProfile(buyer, skipListing=spare.pk, now=now)

            if profile is None:
                continue

            pool = goodFlats(buyer, now=now, skipListing=spare.pk)

            pool = pool.prefetch_related(Prefetch("photos", queryset=photos, to_attr="tastePhotos"))

            candidates = list(pool.order_by("pk"))

            ids = [flat.pk for flat in candidates]

            if spare.pk not in ids:
                continue

            ranked = rankFlats(profile, candidates, limit=len(candidates))

            content_ids = [row[0].pk for row in ranked]

            by_views = sorted(candidates, key=lambda flat: (-flat.viewCount, flat.pk))

            popular_ids = [flat.pk for flat in by_views]

            alike_rank = find_rank(content_ids, spare.pk)

            top_rank = find_rank(popular_ids, spare.pk)

            rankings["contentBased"].append(alike_rank)

            rankings["popularity"].append(top_rank)

            for trial in range(flat_trials):
                mixed = ids.copy()

                rng.shuffle(mixed)

                rankings["random"].append(find_rank(mixed, spare.pk))

            buy_ids.add(buyer.pk)

            folds.append({
                "buyerId": buyer.pk,
                "heldOutId": spare.pk,
                "trainingFlats": len(engaged) - 1,
                "candidates": len(candidates),
                "contentRank": alike_rank,
                "popularityRank": top_rank,
            })

    if not folds:
        raise Exception("No eligible leave-one-out folds in seeded demo data.")

    results = {}

    for name, ranks in rankings.items():
        results[name] = flat_marks(ranks)

    return {
        "method": "Leave each live engaged flat out once per seeded demo buyer; remove all its views, saves, offers and bookings from the profile and exclusions.",
        "buyers": len(buy_ids),
        "folds": len(folds),
        "cutoff": 6,
        "meanCandidateCount": round(sum(fold["candidates"] for fold in folds) / len(folds), 2),
        "randomSeed": flat_seed,
        "randomTrialsPerFold": flat_trials,
        "photoModelVersion": MODEL_VERSION,
        "results": results,
        "foldDetails": folds,
    }

In [5]:
def save_flats(result):
    from storefront import recommend_flats as recs

    clue_names = [
        "VIEW_WEIGHT", "VIEW_CAP", "OLD_VIEW_DAYS", "OLD_VIEW_FACTOR",
        "SAVE_WEIGHT", "OFFER_WEIGHT", "BOOKING_WEIGHT", "SEARCH_WEIGHT",
    ]

    mark_names = [
        "TOWN_WEIGHT", "TYPE_WEIGHT", "PRICE_WEIGHT", "SIZE_WEIGHT",
        "STOREY_WEIGHT", "LEASE_WEIGHT", "PHOTO_WEIGHT", "FAIR_WEIGHT",
    ]

    result["signalWeights"] = {name: getattr(recs, name) for name in clue_names}

    result["scoreWeights"] = {name: getattr(recs, name) for name in mark_names}

    (report_folder / "flat_recs_eval.json").write_text(json.dumps(result, indent=2) + "\n")

    lines = [
        "# Flat recommendations", "",
        "Run the flats check in model_checks.ipynb with the project virtual environment. The script makes a SQLite backup in /tmp, migrates it, regenerates demo activity, and never writes to the source database. Settings are loaded without reading .env; seed valuations reuse stored price bands rather than loading a model.", "",
        f"Leave-one-out uses {result['buyers']} seeded demo buyers with at least three engaged flats and {result['folds']} folds. Each live engaged flat is held out once; all of that buyer's signals for it are removed before building the profile and candidate exclusions. All methods use the same public, live pool with own, other saved/offered/booked and recently viewed flats excluded. Mean pool size: {result['meanCandidateCount']}. Ties use ascending listing ID.", "",
        "Views weigh 1 each, capped at 3 per flat and halved after 30 days. A save weighs 3, an offer 5, a non-cancelled viewing booking 4, and each active saved search 2. Repeated saves across collections and repeated offers/bookings count once per flat per signal. Signals add across kinds. Cold start requires two distinct engaged flats, or a saved search and a viewed flat.", "",
        "The profile contains weighted towns, flat types, log-price mean/spread, size, three-floor storey bands, remaining lease and mean normalised current-version stored CLIP photo vectors. Search filters add soft town/type/price/size/lease preferences. Score weights: town 3, type 2.5, log-price Gaussian 2, size 1, storey 0.5, lease 0.75, photo cosine 2, fair-range bonus 0.2. Minimum Gaussian spreads: log-price 0.2, size 25 sqm, storey 2 bands, lease 15 years. Missing photos are skipped.", "",
        "| Method | Hit-rate@6 | MRR |", "| --- | ---: | ---: |",
    ]

    for key, name in (("contentBased", "Content based"), ("popularity", "Popularity (viewCount)"), ("random", "Random")):
        scores = result["results"][key]

        lines.append(f"| {name} | {scores['hitRateAt6']:.4f} | {scores['mrr']:.4f} |")

    lines.extend(["", f"Random uses {flat_trials} seeded shuffles per fold (seed {flat_seed}). MRR uses the full candidate ranking; the JSON also includes MRR@6, which assigns zero outside the first six.", "",
                  "Limitations: small demo data, synthetic activity and a small candidate pool make hit-rate easy to saturate. Folds for the same buyer overlap and are not independent. Seed activity is broad rather than a measured preference dataset; these results do not establish better buyer outcomes. This is content-based ranking with fixed weights, no trained recommendation model. Existing photo embeddings are optional and never recomputed. Popularity uses the existing global viewCount. This is an offline association test, not a chronological or causal evaluation. Saved searches remain as independent preferences during holdout. Cache lasts ten minutes and activity signatures invalidate it after views, saves, offers, bookings or saved-search changes.", ""])

    (report_folder / "RECS.md").write_text("\n".join(lines))


def try_flats(database):
    use_django("flats", rerun=False)

    source = Path(database).resolve()

    hash_one = hashlib.sha256(source.read_bytes()).hexdigest()

    with tempfile.TemporaryDirectory(prefix="flat-recs-eval-", dir="/tmp") as folder:
        copied = Path(folder) / "db.sqlite3"

        with sqlite3.connect(source.as_uri() + "?mode=ro", uri=True) as original:
            with sqlite3.connect(copied) as target:
                original.backup(target)

        flat_setup(copied, Path(folder) / "media")

        from django.core.management import call_command
        from storefront.models import Listing

        call_command("migrate", interactive=False, stdout=StringIO())

        bands = {}

        for flat in Listing.objects.order_by("pk"):
            low = flat.bandLow or int(flat.askingPrice * 0.9)

            worth = flat.estimatedPrice or flat.askingPrice

            high = flat.bandHigh or int(flat.askingPrice * 1.1)

            bands[flat.town] = (low, worth, high)

        def saved_band(town, *args, **kwargs):
            return bands.get(town, (450000, 500000, 550000))

        with patch("storefront.management.commands.seed_activity.band", side_effect=saved_band):
            call_command("seed_activity", stdout=StringIO())

        result = rate_flats()

        result["sourceDatabaseUnchanged"] = hashlib.sha256(source.read_bytes()).hexdigest() == hash_one

        if not result["sourceDatabaseUnchanged"]:
            raise Exception("Source database changed during evaluation.")

        save_flats(result)

        summary = {}

        for key in ("buyers", "folds", "meanCandidateCount", "results", "sourceDatabaseUnchanged"):
            summary[key] = result[key]

        print(json.dumps(summary, indent=2))

    return result

In [6]:
def room_kind(room):
    if "bedroom" in room:
        return "bedroom"

    if "toilet" in room:
        return "bathroom"

    return room


def find_tools(model_info):
    import numpy
    from storefront.models import Listing, ListingPhoto

    path = project_folder / "storefront" / "lens.py"

    functions = {"validEmbedding", "closeness", "similarFlats"}

    constants = {"RESULTS", "MIN_USEFUL_SIMILARITY", "CLOSENESS"}

    nodes = []

    for node in ast.parse(path.read_text()).body:
        if isinstance(node, ast.FunctionDef) and node.name in functions:
            nodes.append(node)
        elif isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id in constants:
                    nodes.append(node)

                    break

    namespace = {
        "numpy": numpy, "math": math, "Real": Real,
        "Listing": Listing, "ListingPhoto": ListingPhoto,
        "EMBEDDING_SIZE": model_info["embeddingSize"],
        "MODEL_VERSION": model_info["photoModelVersion"],
    }

    exec(compile(ast.Module(body=nodes, type_ignores=[]), str(path), "exec"), namespace)

    return namespace


def pic_marks(rows, key):
    count = len(rows)

    scores = {
        "queries": count,
        "precisionAt1": None,
        "precisionAt5": None,
        "meanReciprocalRank": None,
        "resultCoverage": None,
    }

    if count:
        scores["precisionAt1"] = sum(row[key]["precisionAt1"] for row in rows) / count

        scores["precisionAt5"] = sum(row[key]["precisionAt5"] for row in rows) / count

        scores["meanReciprocalRank"] = sum(row[key]["reciprocalRank"] for row in rows) / count

        scores["resultCoverage"] = sum(bool(row[key]["returned"]) for row in rows) / count

    return scores


def score_hits(matches, room):
    relevant = [room_kind(match["photo"].room) == room_kind(room) for match in matches]

    rank_score = 0.0

    for rank, same in enumerate(relevant, 1):
        if same:
            rank_score = 1 / rank

            break

    return {
        "returned": len(matches),
        "precisionAt1": float(bool(relevant) and relevant[0]),
        "precisionAt5": sum(relevant[:5]) / 5,
        "reciprocalRank": rank_score,
    }

In [7]:
# Check photo search results
def check_photos(database):
    database = Path(database).resolve(strict=True)

    check = f"photos with {database}"

    run_one = use_django(check)

    if str(training_folder) not in sys.path:
        sys.path.insert(0, str(training_folder))

    from write_manifest import file_hash, photoIdentity

    if run_one:
        sys.path.insert(0, str(project_folder))

        import django
        from django.conf import settings

        settings.configure(
            INSTALLED_APPS=["django.contrib.auth", "django.contrib.contenttypes", "storefront.StorefrontConfig"],
            DATABASES={"default": {
                "ENGINE": "django.db.backends.sqlite3",
                "NAME": database.as_uri() + "?mode=ro",
                "OPTIONS": {"uri": True},
            }},
            USE_TZ=True,
        )

        django.setup()

        django_state["check"] = check

    from django.db import connection
    from storefront.models import ListingPhoto

    with connection.cursor() as cursor:
        cursor.execute("PRAGMA query_only = ON")

    ai_info = photoIdentity()

    helpers = find_tools(ai_info)

    photos = list(ListingPhoto.objects.filter(
        embedding__isnull=False, modelVersion=ai_info["photoModelVersion"],
    ).select_related("listing").order_by("pk"))

    valid = [photo for photo in photos if helpers["validEmbedding"](photo.embedding) is not None]

    cutoff = helpers["MIN_USEFUL_SIMILARITY"]

    rows = []

    frozen = []

    started = time.perf_counter()

    for photo in valid:
        helpers["MIN_USEFUL_SIMILARITY"] = -math.inf

        ranked = helpers["similarFlats"](photo.embedding, photo.listing)

        if ranked and ranked[0]["score"] >= cutoff:
            served = ranked
        else:
            served = []

        matches = []

        for match in ranked:
            matches.append({
                "photoId": match["photo"].pk,
                "listingId": match["flat"].pk,
                "room": match["photo"].room,
                "similarity": match["score"],
            })

        rows.append({
            "photoId": photo.pk,
            "listingId": photo.listing_id,
            "room": photo.room,
            "roomType": room_kind(photo.room),
            "ranking": score_hits(ranked, photo.room),
            "served": score_hits(served, photo.room),
            "matches": matches,
        })

        frozen.append([photo.pk, photo.listing_id, photo.room, photo.embedding, photo.modelVersion])

    helpers["MIN_USEFUL_SIMILARITY"] = cutoff

    by_room = {}

    for room in sorted({row["roomType"] for row in rows}):
        room_rows = [row for row in rows if row["roomType"] == room]

        by_room[room] = {
            "ranking": pic_marks(room_rows, "ranking"),
            "served": pic_marks(room_rows, "served"),
        }

    report = {
        "evaluatedAt": datetime.now(timezone.utc).isoformat(),
        "database": database.name,
        "databaseSha256": file_hash(database),
        "querySnapshotSha256": hashlib.sha256(json.dumps(frozen, separators=(",", ":")).encode()).hexdigest(),
        "photoModelVersion": ai_info["photoModelVersion"],
        "rankingCodeSha256": file_hash(project_folder / "storefront" / "lens.py"),
        "method": "Execute unchanged similarFlats, validEmbedding and closeness functions extracted via AST to avoid importing photo heads or vision models; minimal Django settings never read .env; read-only SQLite URI and query_only.",
        "queryScope": "Every stored valid current-version photo, regardless of source listing status; skip its entire listing.",
        "candidateScope": "Serving helper: current-version valid photos of live listings whose sellers are active; best photo per flat; up to 12 flats.",
        "relevance": "Seller-declared room type; all bedroom slots combined as bedroom and toilet slots as bathroom; other slots unchanged.",
        "metricDefinitions": "Macro average over all queries; P@5 divides by 5 even with fewer results; empty results score zero. MRR is truncated at serving RESULTS (12). Ranking ignores only the no-close-match threshold; served applies it. Coverage is the fraction of queries with any served results.",
        "minimumUsefulSimilarity": cutoff,
        "currentVersionPhotos": len(photos),
        "invalidEmbeddingsSkipped": len(photos) - len(valid),
        "ranking": pic_marks(rows, "ranking"),
        "served": pic_marks(rows, "served"),
        "noCloseMatchesQueries": sum(not row["served"]["returned"] for row in rows),
        "perRoom": by_room,
        "elapsedSeconds": time.perf_counter() - started,
        "limitations": [
            "Room labels are seller-declared upload slots, not independently judged visual relevance.",
            "The demo set is small and may repeat photos across flats; no duplicate-photo exclusion is applied because serving does not apply one.",
            "This measures stored-embedding retrieval only, not upload inference latency, visual style, room-slot correctness or real-user utility.",
        ],
        "queries": rows,
    }

    (report_folder / "lens_eval.json").write_text(json.dumps(report, indent=2) + "\n")

    summary = {}

    for key in ["currentVersionPhotos", "ranking", "served", "noCloseMatchesQueries"]:
        summary[key] = report[key]

    print(json.dumps(summary, indent=2))

    connection.close()

    return report

In [8]:
not_flat_labels = {"outdoor", "not_flat"}

label_groups = {
    "living_room": "living_room", "living room": "living_room", "dining_room": "living_room",
    "kitchen": "kitchen", "bedroom": "bedroom", "master_bedroom": "bedroom",
    "bathroom": "toilet", "toilet": "toilet",
    "balcony": "balcony", "service yard": "service_yard", "service_yard": "service_yard",
    "window view": "window_view", "window_view": "window_view", "view": "window_view",
}

override_labels = {"balcony", "service yard", "window view"}


def slot_group(slot):
    if not slot:
        return None

    if "bedroom" in slot:
        return "bedroom"

    if "toilet" in slot:
        return "toilet"

    return label_groups.get(slot, slot)


def count_rate(hits, total):
    rate = None

    if total:
        rate = round(hits / total, 4)

    return {"count": hits, "of": total, "rate": rate}


def kappa(first, second):
    shared = [key for key in first if key in second]

    if not shared:
        return None

    observed = sum(first[key] == second[key] for key in shared) / len(shared)

    first_counts = Counter(first[key] for key in shared)

    second_counts = Counter(second[key] for key in shared)

    expected = sum(first_counts[label] * second_counts[label] for label in first_counts) / len(shared) ** 2

    if expected < 1:
        return round((observed - expected) / (1 - expected), 4)

    return None


def score_tags(labels, photos, review_at):
    used = {key: value for key, value in labels.items() if value != "unsure" and key in photos}

    negatives = [key for key, value in used.items() if value in not_flat_labels]

    rooms = [key for key, value in used.items() if value not in not_flat_labels and value != "other_room"]

    model = {key: photos[key]["model"] for key in used}

    def held(key):
        return model[key]["refused"] or (model[key]["irrelevant"] or 0) >= review_at

    refused_negatives = sum(model[key]["refused"] for key in negatives)

    held_negatives = sum(held(key) for key in negatives)

    refused_rooms = sum(model[key]["refused"] for key in rooms)

    room_hits = sum(label_groups.get(model[key]["label"]) == used[key] for key in rooms)

    overrides = [key for key in used if model[key]["label"] in override_labels]

    override_hits = sum(label_groups.get(model[key]["label"]) == used[key] for key in overrides)

    slotted = [key for key in rooms if photos[key]["sellerSlot"]]

    slot_hits = sum(slot_group(photos[key]["sellerSlot"]) == used[key] for key in slotted)

    return {
        "photosLabelled": len(used),
        "notAFlatPhotos": len(negatives),
        "notAFlatRefused": count_rate(refused_negatives, len(negatives)),
        "notAFlatRefusedOrHeld": count_rate(held_negatives, len(negatives)),
        "flatPhotosWronglyRefused": count_rate(refused_rooms, len(rooms)),
        "roomTypeAgreement": count_rate(room_hits, len(rooms)),
        "overrideLabels": {
            "photos": len(overrides),
            "humanSaid": dict(Counter(used[key] for key in overrides)),
            "agreement": count_rate(override_hits, len(overrides)),
        },
        "sellerSlotAgreement": count_rate(slot_hits, len(slotted)),
    }


def try_labels(folder, report_file):
    folder = Path(folder)

    data = json.loads((folder / "photos.json").read_text())

    photos = {photo["id"]: photo for photo in data["photos"]}

    files = sorted(folder.glob("labels-*.json"))

    if not files:
        raise Exception("No labels-*.json files in human_labels/. Label the photos first.")

    label_sets = [json.loads(path.read_text()) for path in files]

    report = {
        "modelVersion": data["modelVersion"],
        "photos": len(photos),
        "labellers": [],
        "method": "Each person labelled the shuffled photos alone without seeing the seller's slot or the model's answer. Room types are compared in families (all bedrooms, all toilets). Not a flat means outdoors or not a photo of a flat.",
    }

    for item in label_sets:
        scores = score_tags(item["labels"], photos, data["reviewAt"])

        report["labellers"].append({"name": item["labeller"], "labelledAt": item["labelledAt"], **scores})

    if len(label_sets) >= 2:
        first = label_sets[0]["labels"]

        second = label_sets[1]["labels"]

        shared = [key for key in first if key in second]

        same = sum(first[key] == second[key] for key in shared)

        report["agreementBetweenLabellers"] = {
            "cohensKappa": kappa(first, second),
            "sameLabel": count_rate(same, len(shared)),
        }

        agreed = {key: value for key, value in first.items() if second.get(key) == value}

        report["agreedLabels"] = score_tags(agreed, photos, data["reviewAt"])

    Path(report_file).write_text(json.dumps(report, indent=2) + "\n")

    print(json.dumps(report.get("agreedLabels") or report["labellers"][0], indent=1))

    return report

In [9]:
interior_folder = project_folder / "assets" / "seed" / "photos"

estate_folder = project_folder / "assets" / "static" / "storefront" / "estate"

sample_flat = {
    "town": "ANG MO KIO", "flatType": "4 ROOM", "flatModel": "New Generation",
    "floorAreaSqm": 91, "storey": 5, "leaseYear": 1984, "postal": "560474", "block": "474",
}

sample_photos = [
    ("living room photo in the Living room slot", interior_folder / "hdb-460204-08-living_room.jpg", "living_room"),
    ("kitchen photo in the Master bedroom slot", interior_folder / "hdb-460204-08-kitchen.jpg", "master_bedroom"),
    ("outdoor estate photo in the Kitchen slot", estate_folder / "jurongwest.jpg", "kitchen"),
    ("app icon (not a photo of a flat) in the Bedroom 2 slot", project_folder / "assets" / "static" / "storefront" / "icons" / "icon-512.png", "bedroom_2"),
]


def time_call(work):
    start = time.perf_counter()

    value = work()

    return value, round(time.perf_counter() - start, 3)


def try_price(valuation):
    (low, worth, high), seconds = time_call(lambda: valuation.band(**sample_flat))

    asks = [
        ("asking at the estimate", int(round(worth, -3))),
        ("asking 25% above the estimate", int(round(worth * 1.25, -3))),
        ("asking 25% below the estimate", int(round(worth * 0.75, -3))),
    ]

    cases = []

    for name, asking in asks:
        flag = valuation.flagReason(asking, low, worth, high)

        cases.append({
            "case": name,
            "askingPrice": asking,
            "flag": flag,
            "verdict": valuation.verdict(asking, low, worth, high)["text"],
            "listingStatusOnSubmit": "review" if flag else "live (if photos and documents also pass)",
        })

    return {
        "input": sample_flat,
        "modelVersion": valuation.VERSION,
        "band": {"low": round(low), "estimate": round(worth), "high": round(high)},
        "predictionSeconds": seconds,
        "cases": cases,
    }


def judge_pic(photocheck, result, slot):
    if result and result.get("irrelevant") is not None and result["irrelevant"] >= photocheck.BLOCK_AT:
        return "refused with a reason", []

    reasons = photocheck.reviewReasons(result)

    warning = photocheck.slotWarning(result, slot) if result else ""

    if warning:
        reasons.append(warning)

    if reasons:
        return "listing held for staff review", reasons

    return "accepted", reasons


def try_photos(photocheck, pretrained):
    images = [path.read_bytes() for name, path, slot in sample_photos]

    results, seconds = time_call(lambda: photocheck.checkPhotos(images))

    cases = []

    for (name, path, slot), result in zip(sample_photos, results):
        choice, reasons = judge_pic(photocheck, result, slot)

        details = result or {}

        cases.append({
            "case": name,
            "file": str(path.relative_to(project_folder)),
            "slot": slot,
            "label": details.get("label"),
            "roomLabel": details.get("roomLabel"),
            "roomConfidence": details.get("roomConfidence"),
            "notRoomScore": details.get("irrelevant"),
            "aiImageScore": details.get("aiGenerated"),
            "decision": choice,
            "reasons": reasons,
        })

    return {
        "modelInstalled": photocheck.isOn(),
        "identity": pretrained.verifyModel(pretrained.CLIP_NAME),
        "thresholds": {
            "refuseAt": photocheck.BLOCK_AT,
            "reviewAt": photocheck.REVIEW_AT,
            "slotWarningAt": photocheck.SLOT_CONFIDENCE,
        },
        "photos": len(images),
        "checkSeconds": seconds,
        "cases": cases,
    }


def try_packs(Listing, recommendPackage):
    homes = list(Listing.objects.filter(status__in=["live", "review"]).order_by("pk")[:5])

    cases = []

    for listing in homes:
        hint, seconds = time_call(lambda: recommendPackage(listing))

        if hint:
            hint_name = hint["product"].name

            reason = hint["reason"]
        else:
            hint_name = None

            reason = "no paid package suggested (free option predicted, active package, or model unavailable)"

        cases.append({
            "listing": f"{listing.town.title()} {listing.flatType}",
            "status": listing.status,
            "suggestion": hint_name,
            "reason": reason,
            "seconds": seconds,
        })

    return {"listingsChecked": len(cases), "cases": cases}


def check_app(label, report_file):
    if not label:
        raise Exception("Set app_label before running the app check.")

    report_file = Path(report_file)

    run_one = use_django("app")

    os.chdir(project_folder)

    if run_one:
        sys.path.insert(0, str(project_folder))

        os.environ.setdefault("DJANGO_SETTINGS_MODULE", "flatfair.settings")

        import django

        django.setup()

        django_state["check"] = "app"

    import pretrained
    from sellerpanel import photocheck
    from sellerpanel.recommend import bestPack
    from storefront import valuation
    from storefront.models import Listing

    if report_file.is_file():
        report = json.loads(report_file.read_text())
    else:
        report = {}

    report[label] = {
        "ranAt": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "python": platform.python_version(),
        "system": platform.system(),
        "project": project_folder.name,
        "price": try_price(valuation),
        "photos": try_photos(photocheck, pretrained),
        "packages": try_packs(Listing, bestPack),
    }

    report_file.write_text(json.dumps(report, indent=2) + "\n")

    print(json.dumps(report[label]["photos"]["cases"], indent=1)[:1500])

    return report[label]

In [10]:
def run_check(choice):
    if choice == "flats":
        return try_flats(flat_database)

    if choice == "photos":
        return check_photos(photo_database)

    if choice == "labels":
        return try_labels(label_folder, label_report_file)

    if choice == "app":
        return check_app(app_label, app_report_file)

    raise Exception("check_choice must be 'flats', 'photos', 'labels' or 'app'.")

In [11]:
from concurrent.futures import ThreadPoolExecutor

report_folder.mkdir(parents=True, exist_ok=True)

with ThreadPoolExecutor(max_workers=1) as worker:
    check_result = worker.submit(run_check, check_choice).result()

print({"check": check_choice, "short_run": short_run, "source_database_unchanged": check_result.get("sourceDatabaseUnchanged")})

{
  "buyers": 6,
  "folds": 66,
  "meanCandidateCount": 47581.26,
  "results": {
    "contentBased": {
      "hitRateAt6": 0.333333,
      "mrr": 0.249878,
      "mrrAt6": 0.243687
    },
    "popularity": {
      "hitRateAt6": 0.075758,
      "mrr": 0.075785,
      "mrrAt6": 0.075758
    },
    "random": {
      "hitRateAt6": 0.0,
      "mrr": 0.000195,
      "mrrAt6": 0.0
    }
  },
  "sourceDatabaseUnchanged": true
}
{'check': 'flats', 'short_run': True, 'source_database_unchanged': True}
